In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# データの読み込み
matrix = pd.read_pickle("cache/matrix.pkl")
test_clean = pd.read_pickle("cache/test_clean.pkl")
print("Loaded matrix:", matrix.shape, "test_clean:", test_clean.shape)

## データセットの分割
X_train = matrix[matrix.date_block_num < 33].drop(['item_cnt_month'], axis=1)
Y_train = matrix[matrix.date_block_num < 33]['item_cnt_month']
X_valid = matrix[matrix.date_block_num == 33].drop(['item_cnt_month'], axis=1)
Y_valid = matrix[matrix.date_block_num == 33]['item_cnt_month']
X_test = matrix[matrix.date_block_num == 34].drop(['item_cnt_month'], axis=1)

model = XGBRegressor(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.03,
    min_child_weight=300,
    colsample_bytree=0.8,
    subsample=0.8,
    eval_metric="rmse",
    early_stopping_rounds=10,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, Y_train,
    eval_set=[(X_valid, Y_valid)],
    verbose=True
)

pred_valid = model.predict(X_valid)
mse = mean_squared_error(Y_valid, pred_valid) 
rmse = np.sqrt(mse)
print(f"Validation RMSE: {rmse:.5f}")

# 予測と提出ファイル作成
y_test_pred = np.clip(model.predict(X_test), 0, 20)

pred34 = (
    matrix[matrix['date_block_num'] == 34][['shop_id', 'item_id']]
    .copy()
    .assign(item_cnt_month=y_test_pred)
)

submission = (
    test_clean
    .merge(pred34, on=['shop_id', 'item_id'], how='left')
    .set_index('ID')[['item_cnt_month']]
)

# 念のためチェック
assert submission['item_cnt_month'].isna().sum() == 0, "予測に欠損があります。"

# クリップ（課題の通例どおり 0〜20 に収める）
submission['item_cnt_month'] = submission['item_cnt_month'].clip(0, 20)

submission.to_csv('submission_regression.csv')
print("Saved submission_regression.csv")
submission.head()

Loaded matrix: (11128004, 13) test_clean: (214200, 3)
[0]	validation_0-rmse:1.12977
[1]	validation_0-rmse:1.12072
[2]	validation_0-rmse:1.11161
[3]	validation_0-rmse:1.10298
[4]	validation_0-rmse:1.09442
[5]	validation_0-rmse:1.08654
[6]	validation_0-rmse:1.07934
[7]	validation_0-rmse:1.07382
[8]	validation_0-rmse:1.06930
[9]	validation_0-rmse:1.06437
[10]	validation_0-rmse:1.05992
[11]	validation_0-rmse:1.05406
[12]	validation_0-rmse:1.04811
[13]	validation_0-rmse:1.04473
[14]	validation_0-rmse:1.03971
[15]	validation_0-rmse:1.03510
[16]	validation_0-rmse:1.03043
[17]	validation_0-rmse:1.02614
[18]	validation_0-rmse:1.02338
[19]	validation_0-rmse:1.01944
[20]	validation_0-rmse:1.01583
[21]	validation_0-rmse:1.01245
[22]	validation_0-rmse:1.00917
[23]	validation_0-rmse:1.00617
[24]	validation_0-rmse:1.00292
[25]	validation_0-rmse:1.00022
[26]	validation_0-rmse:0.99763
[27]	validation_0-rmse:0.99507
[28]	validation_0-rmse:0.99287
[29]	validation_0-rmse:0.99073
[30]	validation_0-rmse:0.9

,item_cnt_month
ID,
0,0.570009
1,0.096559
2,1.224061
3,0.232021
4,0.893781
